In [5]:
"""
Ensemble Machine Learning for CLAMP-based Paleoclimate Reconstruction

This script implements the methodology described in the manuscript:
- Reads modern calibration data (393 sites) and fossil data (73 assemblages)
- Trains five models: Random Forest, AdaBoost, XGBoost, CatBoost, and a Stacking ensemble
- Evaluates performance via repeated 10-fold cross-validation (R², RMSE, MAE, MAPE)
- Interprets feature importance using SHAP
- Reconstructs paleoclimate from fossil leaf assemblages

Author: Based on provided methods
Date: 2025
"""

import numpy as np
import pandas as pd
import warnings
from sklearn.base import clone
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RepeatedKFold, cross_val_predict, cross_validate
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

import xgboost as xgb
import catboost as cb
import shap

# ----------------------------------------------------------------------
# 1. Load datasets
# ----------------------------------------------------------------------
file_path = "Global378+Tibet15+Fossil73_更正.xlsx"

# Modern calibration dataset (393 sites: 378 global + 15 Tibetan Plateau)
df_modern = pd.read_excel(file_path, sheet_name="plants")
# Fossil dataset (73 assemblages)
df_fossil = pd.read_excel(file_path, sheet_name="fossils")

print(f"Modern dataset shape: {df_modern.shape}")
print(f"Fossil dataset shape: {df_fossil.shape}")

# ----------------------------------------------------------------------
# 2. Define leaf physiognomic traits (31 features) and climate targets (7 variables)
# ----------------------------------------------------------------------
feature_cols = [
    'Lobed', 'No Teeth', 'Teeth Regul', 'Teeth Close', 'Teeth Round', 'Teeth Acute', 'Teeth Compo',
    'Nanophyl', 'Leptophyl', 'Leptophyl 2', 'Microphyl 1', 'Microphyl 2', 'Microphyl 3',
    'Mesophyl 1', 'Mesophyl 2', 'Mesophyl 3', 'Apex Emargi', 'Apex Round', 'Apex Acute', 'Apex Attenu',
    'Base Cordat', 'Base Round', 'Base Acute', 'L:W <1:1', 'L:W 1-2:1', 'L:W 2-3:1', 'L:W 3-4:1',
    'L:W >4:1', 'Obovate', 'Elliptic', 'Ovate'
]

target_cols = ['MAT', 'WMMT', 'CMMT', 'GROWSEAS', 'RH', 'SH', 'ENTHAL']

# Check availability
missing_features = [c for c in feature_cols if c not in df_modern.columns]
missing_targets = [c for c in target_cols if c not in df_modern.columns]
if missing_features:
    raise KeyError(f"Missing features: {missing_features}")
if missing_targets:
    raise KeyError(f"Missing targets: {missing_targets}")

# ----------------------------------------------------------------------
# 3. Data cleaning (drop any rows with missing values)
# ----------------------------------------------------------------------
modern_features = df_modern[feature_cols].copy()
modern_targets = df_modern[target_cols].copy()
mask = modern_features.isna().any(axis=1) | modern_targets.isna().any(axis=1)
if mask.any():
    print(f"Dropping {mask.sum()} rows with missing values in modern dataset.")
    modern_features = modern_features[~mask]
    modern_targets = modern_targets[~mask]

print(f"Clean modern dataset: {modern_features.shape[0]} samples")

# For fossil data: keep only available features (should be same columns)
fossil_features = df_fossil[feature_cols].copy()
if fossil_features.isna().any().any():
    print("Warning: Fossil dataset contains missing values. Filling with column median.")
    fossil_features = fossil_features.fillna(fossil_features.median())

# ----------------------------------------------------------------------
# 4. Define evaluation metrics
# ----------------------------------------------------------------------
def mean_absolute_percentage_error(y_true, y_pred, eps=1e-6):
    """Mean absolute percentage error, safe for near-zero values."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    # Avoid division by zero
    denominator = np.abs(y_true) + eps
    return np.mean(np.abs((y_true - y_pred) / denominator)) * 100

def evaluate_model(y_true, y_pred):
    """Compute R², RMSE, MAE, MAPE."""
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

# ----------------------------------------------------------------------
# 5. Define models
# ----------------------------------------------------------------------
# Base models (using default hyperparameters for reproducibility)
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
ada = AdaBoostRegressor(n_estimators=200, random_state=42)
xgb_model = xgb.XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
cat = cb.CatBoostRegressor(n_estimators=200, random_seed=42, verbose=False)

# Stacking ensemble (base learners + linear meta-learner)
stack = StackingRegressor(
    estimators=[('rf', rf), ('ada', ada), ('xgb', xgb_model), ('cat', cat)],
    final_estimator=LinearRegression(),
    cv=5
)

models = {
    'Random Forest': rf,
    'AdaBoost': ada,
    'XGBoost': xgb_model,
    'CatBoost': cat,
    'Stacking': stack
}

# ----------------------------------------------------------------------
# 6. Cross-validation setup
# ----------------------------------------------------------------------
cv = RepeatedKFold(n_splits=10, n_repeats=5, random_state=42)
results = {target: {name: {'R2': [], 'RMSE': [], 'MAE': [], 'MAPE': []}
                    for name in models} for target in target_cols}

print("\nStarting repeated 10-fold cross-validation (5 repeats)...")
for target in target_cols:
    y = modern_targets[target].values
    X = modern_features.values



# 在性能评估循环内部替换
    for name, model in models.items():
        predictions = []
        true_values = []
        for train_idx, test_idx in cv.split(X, y):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            model_clone = clone(model)
            model_clone.fit(X_train, y_train)
            y_pred = model_clone.predict(X_test)
            predictions.extend(y_pred)
            true_values.extend(y_test)
        scores = evaluate_model(true_values, predictions)
    for metric, value in scores.items():
        results[target][name][metric].append(value)

    print(f"Finished {target}")

# Aggregate results across folds/repeats and compute mean ± std
summary = {}
for target in target_cols:
    summary[target] = {}
    for name in models:
        summary[target][name] = {}
        for metric in ['R2', 'RMSE', 'MAE', 'MAPE']:
            mean_val = np.mean(results[target][name][metric])
            std_val = np.std(results[target][name][metric])
            summary[target][name][metric] = (mean_val, std_val)

# Print performance table
print("\n" + "="*80)
print("MODEL PERFORMANCE (mean ± std across 5×10-fold CV)")
print("="*80)
for target in target_cols:
    print(f"\n{target}:")
    print(f"{'Model':<15} {'R²':<12} {'RMSE':<12} {'MAE':<12} {'MAPE(%)':<12}")
    print("-"*60)
    for name in models:
        r2 = f"{summary[target][name]['R2'][0]:.3f}±{summary[target][name]['R2'][1]:.3f}"
        rmse = f"{summary[target][name]['RMSE'][0]:.2f}±{summary[target][name]['RMSE'][1]:.2f}"
        mae = f"{summary[target][name]['MAE'][0]:.2f}±{summary[target][name]['MAE'][1]:.2f}"
        mape = f"{summary[target][name]['MAPE'][0]:.1f}±{summary[target][name]['MAPE'][1]:.1f}"
        print(f"{name:<15} {r2:<12} {rmse:<12} {mae:<12} {mape:<12}")

# Determine best model per target (based on highest R²)
best_models = {}
for target in target_cols:
    best_name = max(models.keys(), key=lambda n: summary[target][n]['R2'][0])
    best_models[target] = best_name
    print(f"\nBest model for {target}: {best_name} (R² = {summary[target][best_name]['R2'][0]:.3f})")

# ----------------------------------------------------------------------
# 7. Final training on full modern dataset and fossil prediction
# ----------------------------------------------------------------------
print("\n" + "="*80)
print("FINAL TRAINING ON FULL MODERN DATASET & FOSSIL PREDICTION")
print("="*80)

fossil_predictions = pd.DataFrame(index=df_fossil.index)
fossil_predictions['Sample'] = df_fossil['Sample']

X_full = modern_features.values
for target in target_cols:
    y_full = modern_targets[target].values
    best_name = best_models[target]
    model = models[best_name]
    model.fit(X_full, y_full)
    preds = model.predict(fossil_features.values)
    fossil_predictions[target] = preds
    print(f"Reconstructed {target} using {best_name}")

# Save fossil predictions to CSV
fossil_predictions.to_csv("fossil_paleoclimate_predictions.csv", index=False)
print("\nFossil predictions saved to 'fossil_paleoclimate_predictions.csv'")

# ----------------------------------------------------------------------
# 8. SHAP analysis for interpretability (using best model for each target)
# ----------------------------------------------------------------------
print("\n" + "="*80)
print("SHAP FEATURE IMPORTANCE ANALYSIS")
print("="*80)

# Use a background subset for SHAP (100 samples for efficiency)
background = shap.sample(X_full, 100, random_state=42)

for target in target_cols:
    best_name = best_models[target]
    model = models[best_name]
    # Retrain model on full data if not already done in previous loop
    model.fit(X_full, modern_targets[target].values)

    # SHAP explainer depends on model type
    if best_name == 'XGBoost':
        explainer = shap.TreeExplainer(model)
    elif best_name == 'CatBoost':
        explainer = shap.TreeExplainer(model)
    elif best_name in ['Random Forest', 'AdaBoost']:
        explainer = shap.TreeExplainer(model)
    elif best_name == 'Stacking':
        # Stacking has no unified tree explainer; fallback to KernelExplainer on final estimator
        # Simplified: use KernelExplainer with the stacking model's predict method
        def stacking_predict(X):
            return model.predict(X)
        explainer = shap.KernelExplainer(stacking_predict, background)
    else:
        explainer = shap.KernelExplainer(model.predict, background)

    # Compute SHAP values for the first 200 samples (to limit computation)
    X_sample = X_full[:200]
    shap_values = explainer.shap_values(X_sample)

    # Mean absolute SHAP value as feature importance
    mean_shap = np.abs(shap_values).mean(axis=0)
    feature_importance = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': mean_shap
    }).sort_values('Importance', ascending=False)

    print(f"\n{target} (model: {best_name}) - Top 5 important traits:")
    print(feature_importance.head(5).to_string(index=False))

    # Optional: save SHAP summary plot (requires matplotlib)
    try:
        import matplotlib.pyplot as plt
        shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, show=False)
        plt.title(f"SHAP summary - {target}")
        plt.tight_layout()
        plt.savefig(f"SHAP_{target}.png", dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  SHAP plot saved as SHAP_{target}.png")
    except Exception as e:
        print(f"  Could not generate SHAP plot: {e}")

print("\nScript finished successfully.")

Modern dataset shape: (393, 47)
Fossil dataset shape: (71, 47)
Clean modern dataset: 393 samples

Starting repeated 10-fold cross-validation (5 repeats)...
Finished MAT
Finished WMMT
Finished CMMT


KeyboardInterrupt: 